In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 5)#đăt kích thước mặt định cho đồ thị

url = "https://raw.githubusercontent.com/selva86/datasets/master/a10.csv"
df = pd.read_csv(url, parse_dates=["date"], index_col="date")
df.index.freq = "MS"
series = df["value"].astype(float)
print(f"Số quan sát: {len(series)}")
print(f"Giai đoạn  : {series.index.min():%m/%Y} – {series.index.max():%m/%Y}")

chia tập data 80/20

In [ ]:
n_total=len(series)
n_train = int(n_total * 0.80)# số dòng dành cho train: int(204 * 0.8) = 163
train = series.iloc[:n_train]# lấy 163 dòng đầu làm tập huấn luyện
test  = series.iloc[n_train:]# lấy phần còn lại (41 dòng) làm tập kiểm thử

print(f"Train: {len(train)} quan sát  ({train.index[0]:%m/%Y} → {train.index[-1]:%m/%Y})")
print(f"Test : {len(test)}  quan sát  ({test.index[0]:%m/%Y} → {test.index[-1]:%m/%Y})")

In [ ]:
# Vẽ đồ thị phân chia train / test để trực quan hóa
plt.figure()
plt.plot(train, label="Train (80%)", color="#1f77b4")   
plt.plot(test,  label="Test  (20%)", color="orange")     
plt.axvline(train.index[-1], color="red", linestyle="--",        
            linewidth=1.2, label="Ranh giới Train/Test")      
plt.title("Phân chia dữ liệu Train / Test")
plt.xlabel("Tháng")
plt.ylabel("Doanh số (triệu AUD)")
plt.legend()
plt.tight_layout()
plt.show()

fit 2 mô hình arima, ghi lại aic và bic

In [ ]:
def fit_arima(p,d,q, train_data):
    model=SARIMAX(train_data,
                   order=(p,d,q),
                   enforce_stationarity=False,#không ép buộc chuỗi phải dùng trước
                   enforce_invertibility=False)#không ép buộc tính khả nghịch
    return model.fit(disp=False)#huấn luyện mô hình

candidate_orders=[(1, 1, 1), (2, 1, 1)]# ARIMA(1,1,1) và ARIMA(2,1,1)
fitted_models={}# dict lưu kết quả huấn luyện của từng mô hình
#Lần lượt fit từng mô hình và in AIC/BIC
for order in candidate_orders:
    p, d, q = order# giải nén bộ tham số
    fitted_models[order]=fit_arima(p, d, q, train)# huấn luyện và lưu lại
    print(f"AIC = {fitted_models[order].aic:.2f}  |  BIC = {fitted_models[order].bic:.2f}\n")

#tạo bảng so sánh AIC/BIC của 2 mô hình
summary_df = pd.DataFrame({
    "Mô hình":[f"ARIMA{o}" for o in candidate_orders],
    "AIC":[round(fitted_models[o].aic, 2) for o in candidate_orders],# giá trị AIC
    "BIC":[round(fitted_models[o].bic, 2) for o in candidate_orders],# giá trị BIC
})
print(summary_df.to_string(index=False))

#chọn mô hình có AIC thấp nhất (AIC thấp mô hình phù hợp)
best_order=min(candidate_orders, key=lambda o: fitted_models[o].aic)
best_fit=fitted_models[best_order]# lưu mô hình tốt nhất vào biến best_fit
print(f"\nMô hình được chọn: ARIMA{best_order} (AIC thấp nhất)")

chẩn đoán phần dư

In [ ]:
residuals = best_fit.resid# lấy vector phần dư (residual) của mô hình tốt nhất
fig, axes = plt.subplots(2, 1, figsize=(13, 9))

#Đồ thị 1: Residual theo thời gian
axes[0].plot(residuals, color="steelblue", linewidth=1.2)# vẽ đường residual
axes[0].axhline(0, color="red", linestyle="--", linewidth=1)# vẽ đường y=0 làm chuẩn
axes[0].set_title(f"Residuals theo thời gian — ARIMA{best_order}")
axes[0].set_ylabel("Residual")

#Đồ thị 2: ACF của residual (kiểm tra có tự tương quan còn sót lại không)
plot_acf(residuals, lags=36, zero=False,# vẽ ACF từ lag 1 đến lag 36 (bỏ lag 0)
         alpha=0.05, ax=axes[1])# dải đáng tin cậy 95%
axes[1].set_title("ACF của Residuals — Kiểm tra nhiễu trắng (White Noise)")

plt.tight_layout()
plt.show()

In [ ]:
#check mean residual có gần 0 ko
mean_resid = residuals.mean()
print(f"Mean của residuals : {mean_resid:.6f}")
if abs(mean_resid) < 0.01:          
    print("Mean ≈ 0 — Phần dư không còn thành phần hệ thống.")
else:                                      
    print("Mean khác 0 đáng kể — Mô hình có thể chưa phù hợp.")

Dự báo và vẽ đồ thị

In [ ]:
n_forecast = len(test)# số bước cần dự báo = độ dài tập test (41 tháng)\
#Dự báo n_forecast bước tiếp theo dựa trên mô hình đã được train
forecast_obj  = best_fit.get_forecast(steps=n_forecast)
forecast_mean = forecast_obj.predicted_mean
forecast_ci   = forecast_obj.conf_int()# khoảng tin cậy 95% (lower, upper)
forecast_mean.index = test.index # gán lại chỉ mục thời gian cho forecast
forecast_ci.index   = test.index# gán lại chỉ mục thời gian cho khoảng tin cậy

#vẽ đồ thị train/test
plt.figure(figsize=(13, 6))
plt.plot(train, label="Train (80%)",      color="#1f77b4", linewidth=1.5)
plt.plot(test,  label="Test — Thực tế",   color="orange",  linewidth=1.5)
plt.plot(forecast_mean, label="Forecast", color="green",   linewidth=1.5,# đường dự báo
         linestyle="--")      
                                             
plt.fill_between(forecast_ci.index,# tô màu khoảng tin cậy 95%
                 forecast_ci.iloc[:, 0],# cận dưới (lower bound)
                 forecast_ci.iloc[:, 1],# cận trên (upper bound)
                 color="green", alpha=0.15, label="95% Confidence Interval")
plt.axvline(train.index[-1], color="red", linestyle="--",# đường phân chia train/test
            linewidth=1, label="Ranh giới Train/Test")

plt.title(f"Dự báo ARIMA{best_order} — Train / Test / Forecast")
plt.xlabel("Tháng")
plt.ylabel("Doanh số (triệu AUD)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

Đánh giá MAE, RMSE, MAPE

In [ ]:
y_true=test.values# mảng giá trị thực tế của tập test 
y_pred=forecast_mean.values# mảng giá trị dự báo tương ứng

#MAE: Trung bình sai số tuyệt đối — đơn vị giống với dữ liệu gốc (triệu AUD)
mae=mean_absolute_error(y_true, y_pred)

#RMSE: Căn bậc hai của trung bình bình phương sai số — phạt nặng hơn với sai số lớn
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# MAPE: Trung bình phần trăm sai số tuyệt đối — dễ hiểu vì tính theo %
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

#In kết quả đánh giá
print(f"Đánh giá mô hình ARIMA{best_order}")
print(f"MAE  : {mae:.4f}")# sai số trung bình tuyệt đối
print(f"RMSE : {rmse:.4f}")# căn bậc hai sai số bình phương trung bình
print(f"MAPE : {mape:.2f}%")# phần trăm sai số trung bình

# Đánh giá tự động dựa trên MAPE
if mape < 10:
    print("MAPE < 10% → Mô hình dự báo TỐT.")
elif mape < 20:
    print("MAPE 10–20% → Mô hình dự báo KHÁ.")
else:
    print("MAPE > 20% → Mô hình yếu — nên nâng cấp lên SARIMA.")